# Data Dharma by Srikanth
## SQL ↔ PySpark Bridge — Part 3
### Aggregations: GROUP BY, COUNT, SUM, AVG, MIN, MAX & HAVING

**PART 1 — FOUNDATIONS ✅**
SELECT, FILTER, DISTINCT, SORT, LIMIT

**PART 2 — TRANSFORMATIONS ✅**
CASE WHEN, CAST, IN, LIKE, NULL, String & Date Functions

**PART 3 — AGGREGATIONS 🚀**
GROUP BY, COUNT, SUM, AVG, MIN, MAX, HAVING

### What does GROUP BY actually do?

```
RAW ORDERS

Houston   100
Dallas    200
Houston   300
Dallas    150
Austin    250

      ↓ GROUP BY city

Houston   → 2 orders → 400 total
Dallas    → 2 orders → 350 total
Austin    → 1 order  → 250 total
```

GROUP BY collects rows that have the same value into groups. Aggregate functions such as COUNT, SUM, and AVG then calculate one summary result for each group.

**Without GROUP BY:** an aggregate function summarizes the entire dataset into one row.
**With GROUP BY:** an aggregate function returns one summary row for each group.

## SECTION 0 — Data Setup

Reusing the exact same `orders` dataset from Part 2 — same rows, same `order_amount` formula, same `customer_name` / `ship_date` / `promo_code` columns. This notebook is independently runnable — you don't need to run Part 2 first.

In [0]:
# from datetime is a module in Python and date is a class in the datetime module
from datetime import date

# from pyspark.sql.types is a module in Python and StructType is a class in it
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    DoubleType,
)

# from pyspark.sql.functions is a module in Python
# each of these is a function in the pyspark.sql.functions module
# Python already has built-in functions named sum, min, max, and round.
# We use aliases such as spark_sum so it is obvious these are PySpark functions, not Python's built-ins.
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    avg,
    min as spark_min,
    max as spark_max,
    round as spark_round,
)

In [0]:
# below is the schema for the orders table
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("ship_date", DateType(), True),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("payment_method", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_amount", DoubleType(), False),
    StructField("promo_code", StringType(), True),
])

In [0]:
# below is the list with sample data
orders_data = [
    (1001, 501, "  raj kumar",      date(2026, 1, 5),  date(2026, 1, 8),  "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 3, 250.00, 20.00, "SAVE10"),
    (1002, 502, "MEENA REDDY ",     date(2026, 1, 6),  date(2026, 1, 8),  "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  2, 150.00, 10.00, None),
    (1003, 503, "Suresh Babu",      date(2026, 1, 7),  None,              "Austin",   "TX", "PENDING",   "PAYPAL",      1, 500.00,  0.00, "WELCOME5"),
    (1004, 504, "Anita Rao",        date(2026, 1, 8),  date(2026, 1, 12), "Chicago",  "IL", "COMPLETED", "UPI",         5,  80.00, 15.00, None),
    (1005, 505, "Kiran Varma",      date(2026, 1, 9),  None,              "New York", "NY", "CANCELLED", "CREDIT_CARD", 4, 120.00,  0.00, None),
    (1006, 501, " Raj Kumar",       date(2026, 1, 10), date(2026, 1, 13), "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 2, 300.00, 25.00, "SAVE10"),
    (1007, 506, "Divya Nair",       date(2026, 1, 11), date(2026, 1, 13), "Dallas",   "TX", "RETURNED",  "DEBIT_CARD",  1, 200.00,  0.00, None),
    (1008, 507, "ramesh iyer",      date(2026, 1, 12), date(2026, 1, 17), "Austin",   "TX", "COMPLETED", "PAYPAL",      3, 100.00, 10.00, "FESTIVE20"),
    (1009, 508, "Priya Sharma",     date(2026, 1, 13), None,              "Chicago",  "IL", "PENDING",   "UPI",         2,  60.00,  5.00, None),
    (1010, 509, "Arjun Menon",      date(2026, 1, 14), date(2026, 1, 17), "New York", "NY", "COMPLETED", "CREDIT_CARD", 6,  90.00, 20.00, "WELCOME5"),
    (1011, 502, "MEENA REDDY",      date(2026, 1, 15), date(2026, 1, 17), "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  4, 175.00, 30.00, None),
    (1012, 510, "Lakshmi Pillai",   date(2026, 1, 16), date(2026, 1, 22), "Houston",  "TX", "COMPLETED", "PAYPAL",      1, 800.00, 50.00, "SAVE10"),
    (1013, 511, "Vikram Rao",       date(2026, 1, 17), None,              "Austin",   "TX", "CANCELLED", "UPI",         2, 250.00,  0.00, None),
    (1014, 512, "Sneha Gupta",      date(2026, 1, 18), date(2026, 1, 22), "Chicago",  "IL", "COMPLETED", "CREDIT_CARD", 3, 220.00, 10.00, "FESTIVE20"),
    (1015, 513, "Karthik Reddy",    date(2026, 1, 19), date(2026, 1, 22), "New York", "NY", "RETURNED",  "DEBIT_CARD",  1, 400.00,  0.00, None),
    (1016, 503, "  Suresh Babu  ",  date(2026, 1, 20), date(2026, 1, 22), "Austin",   "TX", "COMPLETED", "PAYPAL",      5,  60.00,  5.00, "WELCOME5"),
    (1017, 514, "Deepa Krishnan",   date(2026, 1, 21), None,              "Houston",  "TX", "PENDING",   "CREDIT_CARD", 2, 175.00,  0.00, None),
    (1018, 505, "Kiran Varma",      date(2026, 1, 22), date(2026, 1, 25), "New York", "NY", "COMPLETED", "UPI",         3, 210.00, 15.00, "SAVE10"),
]

# create a dataframe with the schema and data
orders_raw_df = spark.createDataFrame(orders_data, schema=orders_schema)

# display the dataframe
display(orders_raw_df)

In [0]:
# create a new column 'order_amount' by multiplying 'quantity' and 'unit_price' and subtracting 'discount_amount'
# round the result to 2 decimal places and cast it to decimal(10,2)

orders_df = orders_raw_df.withColumn(
    "order_amount",
    spark_round((col("quantity") * col("unit_price")) - col("discount_amount"), 2).cast("decimal(10,2)")
)

# display the dataframe
display(orders_df)

In [0]:
# create a temp view 'orders' from the dataframe
orders_df.createOrReplaceTempView("orders")

From this point onward, SQL and PySpark are reading the same data —
SQL through the view `orders`, PySpark through the DataFrame `orders_df`.

## SECTION 1 — COUNT WITHOUT GROUP BY

**Business Requirement:** How many orders do we have?

### 🟨 SQL

In [0]:
%sql
-- Count all orders

SELECT COUNT(*) AS total_orders
FROM orders;

### 🟦 PySpark

In [0]:
# Count all orders

result_df = orders_df.select(
    count("*").alias("total_orders")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`COUNT(*)` ↔ `count("*")`

**A quick but important distinction:** `orders_df.count()` (called directly on the DataFrame, no `.select()`) is a DataFrame **action** — it immediately returns a plain Python integer. `count("*")` used inside `.select()` or `.agg()` is an **aggregation expression** — it builds a column, the way `COUNT(*)` does inside a SQL `SELECT`. For translating SQL aggregations, `count("*")` is the one that maps directly.

**One more important distinction:** `COUNT(*)` counts every row. `COUNT(column)` counts only the **non-NULL** values in that column. Since `promo_code` is `NULL` for orders that didn't use a promo code, this difference is easy to see.

### 🟨 SQL

In [0]:
%sql
-- Count all orders, and separately count only orders that have a promo code

SELECT
    COUNT(*) AS total_orders,
    COUNT(promo_code) AS orders_with_promo
FROM orders;

### 🟦 PySpark

In [0]:
# Count all orders, and separately count only orders that have a promo code

result_df = orders_df.select(
    count("*").alias("total_orders"),
    count("promo_code").alias("orders_with_promo")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`COUNT(*)` ↔ `count("*")` — counts every row
`COUNT(column)` ↔ `count("column")` — counts only non-NULL values in that column

## SECTION 2 — GROUP BY + COUNT

**Business Requirement:** How many orders came from each city?

### 🟨 SQL

In [0]:
%sql
-- Count the number of orders in each city

SELECT
    city,
    COUNT(*) AS order_count
FROM orders
GROUP BY city
ORDER BY city;

### 🟦 PySpark

In [0]:
# Count the number of orders in each city

result_df = (
    orders_df
    .groupBy("city")
    .agg(
        count("*").alias("order_count")
    )
    .orderBy("city")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

| SQL | PySpark |
|---|---|
| `GROUP BY city` | `.groupBy("city")` |
| `COUNT(*)` | `count("*")` |
| `AS order_count` | `.alias("order_count")` |

## SECTION 3 — GROUP BY + SUM

**Business Requirement:** What is the total order amount for each city?

### 🟨 SQL

In [0]:
%sql
-- Calculate total order amount for each city

SELECT
    city,
    SUM(order_amount) AS total_order_amount
FROM orders
GROUP BY city
ORDER BY city;

### 🟦 PySpark

In [0]:
# Calculate total order amount for each city

result_df = (
    orders_df
    .groupBy("city")
    .agg(
        spark_sum("order_amount").alias("total_order_amount")
    )
    .orderBy("city")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`SUM(order_amount)` ↔ `spark_sum("order_amount")`

## SECTION 4 — GROUP BY + AVG

**Business Requirement:** What is the average order amount for each city? Rounded to 2 decimal places so SQL and PySpark display identically.

### 🟨 SQL

In [0]:
%sql
-- Calculate the average order amount for each city

SELECT
    city,
    CAST(ROUND(AVG(order_amount), 2) AS DECIMAL(10,2)) AS avg_order_amount
FROM orders
GROUP BY city
ORDER BY city;

### 🟦 PySpark

In [0]:
# Calculate the average order amount for each city

result_df = (
    orders_df
    .groupBy("city")
    .agg(
        spark_round(avg("order_amount"), 2).cast("decimal(10,2)").alias("avg_order_amount")
    )
    .orderBy("city")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`AVG(order_amount)` ↔ `avg("order_amount")`

Both sides round to 2 decimal places and cast to `decimal(10,2)` — without this, `AVG` can return many decimal places, and SQL and PySpark could display differently even though the underlying value matches.

## SECTION 5 — MIN AND MAX

**Business Requirement:** What are the smallest and largest order amounts in each city?

### 🟨 SQL

In [0]:
%sql
-- Find the smallest and largest order amount in each city

SELECT
    city,
    MIN(order_amount) AS min_order_amount,
    MAX(order_amount) AS max_order_amount
FROM orders
GROUP BY city
ORDER BY city;

### 🟦 PySpark

In [0]:
# Find the smallest and largest order amount in each city

result_df = (
    orders_df
    .groupBy("city")
    .agg(
        spark_min("order_amount").alias("min_order_amount"),
        spark_max("order_amount").alias("max_order_amount")
    )
    .orderBy("city")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`MIN()` ↔ `spark_min()`
`MAX()` ↔ `spark_max()`

## SECTION 6 — MULTIPLE AGGREGATIONS

**Business Requirement:** For each city, show the number of orders, total order amount, average order amount, minimum order amount, and maximum order amount — all at once.

### 🟨 SQL

In [0]:
%sql
-- Summarize order count, total, average, minimum, and maximum order amount for each city

SELECT
    city,
    COUNT(*) AS order_count,
    SUM(order_amount) AS total_order_amount,
    CAST(ROUND(AVG(order_amount), 2) AS DECIMAL(10,2)) AS avg_order_amount,
    MIN(order_amount) AS min_order_amount,
    MAX(order_amount) AS max_order_amount
FROM orders
GROUP BY city
ORDER BY city;

### 🟦 PySpark

In [0]:
# Summarize order count, total, average, minimum, and maximum order amount for each city

result_df = (
    orders_df
    .groupBy("city")
    .agg(
        count("*").alias("order_count"),
        spark_sum("order_amount").alias("total_order_amount"),
        spark_round(avg("order_amount"), 2).cast("decimal(10,2)").alias("avg_order_amount"),
        spark_min("order_amount").alias("min_order_amount"),
        spark_max("order_amount").alias("max_order_amount")
    )
    .orderBy("city")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

SQL: multiple aggregate expressions go inside `SELECT`.
PySpark: multiple aggregate expressions go inside `.agg()`, separated by commas.

**Memory trick:**
`.groupBy()` → **creates the groups**
`.agg()` → **calculates the summary** for each group

## SECTION 7 — GROUP BY MULTIPLE COLUMNS

**Business Requirement:** How many orders are there for each city and order status?

One `GROUP BY` column means one grouping dimension. Multiple `GROUP BY` columns mean one group for every unique **combination** of those columns — for example `Houston + COMPLETED`, `Houston + RETURNED`, and `Dallas + COMPLETED` are each their own group.

### 🟨 SQL

In [0]:
%sql
-- Count orders by city and order status

SELECT
    city,
    order_status,
    COUNT(*) AS order_count
FROM orders
GROUP BY city, order_status
ORDER BY city, order_status;

### 🟦 PySpark

In [0]:
# Count orders by city and order status

result_df = (
    orders_df
    .groupBy("city", "order_status")
    .agg(
        count("*").alias("order_count")
    )
    .orderBy("city", "order_status")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`GROUP BY city, order_status` ↔ `.groupBy("city", "order_status")`

## SECTION 8 — WHERE BEFORE GROUP BY

**Business Requirement:** For COMPLETED orders only, calculate total order amount by city.

```
FILTER ROWS
     ↓
GROUP REMAINING ROWS
     ↓
AGGREGATE EACH GROUP
```

`WHERE` filters individual rows **before** aggregation — in this example, that means `.filter()` runs **before** `.groupBy()`. (This is about the sequence in this specific chain, not a claim that `WHERE` and `.filter()` behave identically in every possible context.)

### 🟨 SQL

In [0]:
%sql
-- Calculate total order amount by city, for completed orders only

SELECT
    city,
    SUM(order_amount) AS total_order_amount
FROM orders
WHERE order_status = 'COMPLETED'
GROUP BY city
ORDER BY city;

### 🟦 PySpark

In [0]:
# Calculate total order amount by city, for completed orders only

result_df = (
    orders_df
    .filter(col("order_status") == "COMPLETED")
    .groupBy("city")
    .agg(
        spark_sum("order_amount").alias("total_order_amount")
    )
    .orderBy("city")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

| SQL | PySpark |
|---|---|
| `WHERE` | `.filter()` — rows first |
| `GROUP BY` | `.groupBy()` |
| `SUM` | `spark_sum()` |

## SECTION 9 — HAVING AFTER GROUP BY

**Business Requirement:** Show only cities whose total order amount is above 1500.

```
WHERE
  ↓
FILTER ROWS
  ↓
GROUP BY
  ↓
AGGREGATE
  ↓
HAVING
  ↓
FILTER GROUPS
```

`WHERE` filters **rows**, before aggregation. `HAVING` filters **grouped/aggregated results**, after aggregation — you can only reference an aggregate like `SUM(order_amount)` in a `HAVING` clause, never in a `WHERE` clause.

**PySpark has no `.having()` method.** `HAVING` translates to `.filter()` again — but this time it comes **after** `.agg()`, not before `.groupBy()`.

### 🟨 SQL

In [0]:
%sql
-- Return only cities whose total order amount exceeds 1500

SELECT
    city,
    SUM(order_amount) AS total_order_amount
FROM orders
GROUP BY city
HAVING SUM(order_amount) > 1500
ORDER BY total_order_amount DESC;

### 🟦 PySpark

In [0]:
# Return only cities whose total order amount exceeds 1500

result_df = (
    orders_df
    .groupBy("city")
    .agg(
        spark_sum("order_amount").alias("total_order_amount")
    )
    .filter(col("total_order_amount") > 1500)
    .orderBy(col("total_order_amount").desc())
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping — the most important one in this notebook:**

`WHERE` → filters **ROWS**, **BEFORE** aggregation
`HAVING` → filters **GROUPS**, **AFTER** aggregation

Same PySpark method, `.filter()`, doing two different jobs depending on where it sits in the chain.

## SECTION 10 — WHERE + GROUP BY + HAVING

**Business Requirement:** For COMPLETED orders only, calculate total order amount by city, and show only cities whose total is above 1000.

### 🟨 SQL

In [0]:
%sql
-- Return completed-order totals by city, keeping only cities above 1000

SELECT
    city,
    SUM(order_amount) AS total_order_amount
FROM orders
WHERE order_status = 'COMPLETED'
GROUP BY city
HAVING SUM(order_amount) > 1000
ORDER BY total_order_amount DESC;

### 🟦 PySpark

In [0]:
# Return completed-order totals by city, keeping only cities above 1000

result_df = (
    orders_df
    .filter(col("order_status") == "COMPLETED")
    .groupBy("city")
    .agg(
        spark_sum("order_amount").alias("total_order_amount")
    )
    .filter(col("total_order_amount") > 1000)
    .orderBy(col("total_order_amount").desc())
)

# display data in the DataFrame result_df
display(result_df)

```
RAW ROWS
   ↓
WHERE / .filter()
   ↓
GROUP BY / .groupBy()
   ↓
SUM / .agg()
   ↓
HAVING / .filter()
   ↓
ORDER BY / .orderBy()
```

This example uses the full chain — exactly the sequence in the diagram above.

## SECTION 11 — COMBINED REAL-WORLD EXAMPLE

**Business Requirement:** For completed orders, create a state-level sales summary — order count, total order amount, average, minimum, and maximum order amount. Keep only states with at least 3 completed orders. Sort highest total order amount first.

### 🟨 SQL

In [0]:
%sql
-- Summarize completed orders by state, keeping only states with at least 3 completed orders

SELECT
    state,
    COUNT(*) AS order_count,
    SUM(order_amount) AS total_order_amount,
    CAST(ROUND(AVG(order_amount), 2) AS DECIMAL(10,2)) AS avg_order_amount,
    MIN(order_amount) AS min_order_amount,
    MAX(order_amount) AS max_order_amount
FROM orders
WHERE order_status = 'COMPLETED'
GROUP BY state
HAVING COUNT(*) >= 3
ORDER BY total_order_amount DESC;

### 🟦 PySpark

In [0]:
# Summarize completed orders by state, keeping only states with at least 3 completed orders

result_df = (
    orders_df
    .filter(col("order_status") == "COMPLETED")
    .groupBy("state")
    .agg(
        count("*").alias("order_count"),
        spark_sum("order_amount").alias("total_order_amount"),
        spark_round(avg("order_amount"), 2).cast("decimal(10,2)").alias("avg_order_amount"),
        spark_min("order_amount").alias("min_order_amount"),
        spark_max("order_amount").alias("max_order_amount")
    )
    .filter(col("order_count") >= 3)
    .orderBy(col("total_order_amount").desc())
)

# display data in the DataFrame result_df
display(result_df)

Same requirement. Same filter → group → aggregate → filter groups → sort chain. Only Texas (`TX`) has 3 or more completed orders in this dataset — Illinois and New York each have 2, so `HAVING` correctly drops them.

## SECTION 12 — VIEWER CHALLENGE

**Pause the video and try this first using SQL, then translate it into PySpark.**

**Requirement:** For `RETURNED` orders — group by city, count the returned orders, and calculate the total returned order amount. Keep only cities where the total returned amount exceeds 300. Sort by total returned amount descending.

**Required output:** `city`, `returned_order_count`, `total_returned_amount`

--- PAUSE HERE ---

### 🟨 SQL SOLUTION

In [0]:
%sql
-- Summarize returned orders by city, keeping only cities above 300 in returned amount

SELECT
    city,
    COUNT(*) AS returned_order_count,
    SUM(order_amount) AS total_returned_amount
FROM orders
WHERE order_status = 'RETURNED'
GROUP BY city
HAVING SUM(order_amount) > 300
ORDER BY total_returned_amount DESC;

### 🟦 PYSPARK SOLUTION

In [0]:
# Summarize returned orders by city, keeping only cities above 300 in returned amount

result_df = (
    orders_df
    .filter(col("order_status") == "RETURNED")
    .groupBy("city")
    .agg(
        count("*").alias("returned_order_count"),
        spark_sum("order_amount").alias("total_returned_amount")
    )
    .filter(col("total_returned_amount") > 300)
    .orderBy(col("total_returned_amount").desc())
)

# display data in the DataFrame result_df
display(result_df)

**RESULT:** Both queries return exactly one row — `New York`, with a returned amount of 400. `Dallas` had a returned order too, but at 200 it doesn't clear the 300 threshold, so `HAVING` correctly excludes it.

## Final Check — Did SQL and PySpark Return the Same Grouped Summary?

We wrote the same aggregation logic in SQL and PySpark, using the Section 11 combined example.

Each row in that result has **six** meaningful values — `state`, `order_count`, `total_order_amount`, `avg_order_amount`, `min_order_amount`, and `max_order_amount`. So instead of comparing a single column, we compare the **complete row** for every group.

### What will we do?

1. Run the SQL version and store the result in `sql_result`
2. Run the PySpark version and store the result in `pyspark_result`
3. Sort both results by `state`, so the comparison is order-sensitive and fair
4. Collect each full row into a Python list
5. Compare the two lists directly

This verifies that every output column matches for every group — not just that the group keys line up.

In [0]:
# Compare SQL and PySpark results to verify every aggregated column matches for every group

# Run the SQL query and store the result as a DataFrame
sql_result = spark.sql("""
    SELECT
        state,
        COUNT(*) AS order_count,
        SUM(order_amount) AS total_order_amount,
        CAST(ROUND(AVG(order_amount), 2) AS DECIMAL(10,2)) AS avg_order_amount,
        MIN(order_amount) AS min_order_amount,
        MAX(order_amount) AS max_order_amount
    FROM orders
    WHERE order_status = 'COMPLETED'
    GROUP BY state
    HAVING COUNT(*) >= 3
    ORDER BY state
""")

# Apply the same logic using PySpark and store the result as a DataFrame
pyspark_result = (
    orders_df
    .filter(col("order_status") == "COMPLETED")
    .groupBy("state")
    .agg(
        count("*").alias("order_count"),
        spark_sum("order_amount").alias("total_order_amount"),
        spark_round(avg("order_amount"), 2).cast("decimal(10,2)").alias("avg_order_amount"),
        spark_min("order_amount").alias("min_order_amount"),
        spark_max("order_amount").alias("max_order_amount")
    )
    .filter(col("order_count") >= 3)
    .orderBy("state")
)

# Trigger execution with collect(), bring each full row to the driver, as a dictionary of column values
sql_rows = [row.asDict() for row in sql_result.collect()]

# Trigger execution with collect(), bring each full PySpark row to the driver, as a dictionary of column values
pyspark_rows = [row.asDict() for row in pyspark_result.collect()]

# Display both full result sets and check whether every column matches for every group
print("SQL rows:     ", sql_rows)
print("PySpark rows: ", pyspark_rows)
print("Same full rows (all columns, order-sensitive):", sql_rows == pyspark_rows)

### Understanding the Result

`row.asDict()` turns one Spark Row into a Python dictionary of `{column_name: value}` — so `sql_rows` and `pyspark_rows` are each a list of dictionaries, one per group.

`sql_rows == pyspark_rows` compares those lists directly. For this to be `True`, every group must appear in the same order, with the same `order_count`, `total_order_amount`, `avg_order_amount`, `min_order_amount`, and `max_order_amount` values — a much stronger check than just matching one identifier column.

If the result is `True` ✅, our SQL and PySpark aggregations produced identical output, group by group, column by column.

To be precise about what this verifies: it confirms the SQL and PySpark **grouped summary** matches exactly, for the groups shown. It doesn't independently re-derive the aggregation from raw rows — it trusts that Spark's SQL engine and DataFrame engine are computing correctly, and checks that the two APIs agree with each other.

## SECTION 14 — FINAL CHEAT SHEET

| SQL | PySpark |
|---|---|
| `COUNT(*)` | `count("*")` |
| `COUNT(column)` | `count("column")` |
| `SUM` | `spark_sum()` |
| `AVG` | `avg()` |
| `MIN` | `spark_min()` |
| `MAX` | `spark_max()` |
| `GROUP BY city` | `.groupBy("city")` |
| `GROUP BY city, status` | `.groupBy("city", "status")` |
| multiple aggregates | `.agg(...)` |
| `WHERE` | `.filter()` **before** `.groupBy()` |
| `HAVING` | `.filter()` **after** `.agg()` |
| `ORDER BY` | `.orderBy()` |

```
RAW ROWS
   ↓
WHERE / .filter()
   ↓
GROUP BY / .groupBy()
   ↓
AGGREGATE / .agg()
   ↓
HAVING / .filter()
   ↓
ORDER BY / .orderBy()
```

**Memory trick, worth repeating:**

`WHERE` → filters **ROWS**, **BEFORE** aggregation
`HAVING` → filters **GROUPS**, **AFTER** aggregation

```
     SQL
      ↕
   PySpark
      ↓
Same Business Logic
```

## Part 3 complete.

Same data. Same business requirement. Different syntax. Same business logic.

**Coming next — Part 4: SQL ↔ PySpark Joins & Multi-Table Data**